In [21]:
from pathlib import Path
from torchvision.transforms import v2
import torch
import torch.nn as nn
import mlflow
import mlflow.pytorch

# ==========================================================
# LOAD classifier model
tracking_uri = Path("../experiments/mlflow.db").resolve()
mlflow.set_tracking_uri(f"sqlite:///{tracking_uri}")
run_id = "531b469fcd5f49ca86d611789277b437"
classifier = mlflow.pytorch.load_model(f"runs:/{run_id}/best_model")
classifier.eval()
# ==========================================================

inference_transform = v2.Compose([
  v2.ToImage(),
  v2.Resize((64,64)),
  v2.ToDtype(torch.float32, scale=True),
  # v2.Normalize(mean=, std=) # depends on which model i choose , where it was trained
])

In [22]:
from itertools import islice
from PIL import Image
# from ultralytics import YOLO
from sahi import AutoDetectionModel, Category
from sahi.predict import get_sliced_prediction, ObjectPrediction
# NOTE: SAHI doesn't like Path objects, it wants string!

CLASS_NAMES = {0:"bud", 1:"flower", 2:"green", 3:"pink", 4:"purple", 5:"blue"}
DEVICE = "cuda:0"
CONFIDENCE_THRESHOLD = 0.26
MODEL_PATH = Path.cwd().parent / "twostage" / "best.pt" 
TEST_IMAGE_PATH = Path.home() / Path("BLUEBERRY_DATA/datasets/dataset-c834a7b7") / "images" / "c834a7b7-4bbd-4791-98c0-9d2d8123d60a_R_00000298.png"
print(MODEL_PATH)
print(TEST_IMAGE_PATH.exists())

test_image = Image.open(TEST_IMAGE_PATH).convert("RGB")
print(test_image)

detection_model = AutoDetectionModel.from_pretrained(
  model_type="ultralytics",
  model_path=str(MODEL_PATH),
  confidence_threshold=CONFIDENCE_THRESHOLD,
  device=DEVICE
)
# detection_model.model # the yolo model

result = get_sliced_prediction(
  image=test_image, # str(TEST_IMAGE_PATH),
  detection_model=detection_model,
  slice_height=640,
  slice_width=640,
  overlap_height_ratio=0.2,
  overlap_width_ratio=0.2,
  postprocess_type="NMS", # defalt: GREEDYNMM
  postprocess_match_metric="IOU", #default:IOS
  postprocess_match_threshold=0.5
)
# print(len(result.object_prediction_list))
# result.object_prediction_list[:3]

for i, pred in islice(enumerate(result.object_prediction_list), 10):
  print(pred)
  bbox = pred.bbox
  print(bbox.to_xyxy()) # xmin, ymin, xman, ymax

  # TODO: add a bit of padding before crop, if detection boxes are too tight
  ''' e.g.
  pad = 10
  x1 = max(0, x1 - pad)
  y1 = max(0, y1 - pad)
  x2 = min(img.width,  x2 + pad)
  y2 = min(img.height, y2 + pad)
  '''
  crop = test_image.crop(bbox.to_xyxy()) # floats e.g. (361.2708435058594, 1532.807876586914, 411.8555908203125, 1584.9158935546875)
  crop_transformed = inference_transform(crop).unsqueeze(0).to(DEVICE)
  print(crop_transformed.shape)

  with torch.no_grad():
    logits = classifier(crop_transformed).detach()
    classifier_probs = nn.Softmax(dim=1)(logits) # (1, n_classes)
    classifier_pred = logits.argmax(dim=1).item()
    classifier_conf = classifier_probs[:, classifier_pred].item()

  updated_objpred_after_classifier = ObjectPrediction(
    bbox=pred.bbox.to_xyxy(), # [pred.bbox.minx, pred.bbox.miny,pred.bbox.maxx,pred.bbox.maxy], # [minx, miny, maxx, maxy]
    category_id=classifier_pred,
    category_name=CLASS_NAMES[classifier_pred],
    score=pred.score,
  )

  # update ObjectPrediction with the new object containing the classifier category
  result.object_prediction_list[i] = updated_objpred_after_classifier
  # pred = updated_pred_after_classifier # this doesn't take effect

/home/kpetrakis/ml-sandbox/twostage/best.pt
True
<PIL.Image.Image image mode=RGB size=1536x2048 at 0x7450341B19C0>
Performing prediction on 12 slices.


ObjectPrediction<
    bbox: BoundingBox: <(361.2708435058594, 1532.807876586914, 411.8555908203125, 1584.9158935546875), w: 50.584747314453125, h: 52.10801696777344>,
    mask: None,
    score: PredictionScore: <value: 0.8999278545379639>,
    category: Category: <id: 0, name: item>>
[361.2708435058594, 1532.807876586914, 411.8555908203125, 1584.9158935546875]
torch.Size([1, 3, 64, 64])
ObjectPrediction<
    bbox: BoundingBox: <(321.7101135253906, 1594.4351806640625, 368.5424499511719, 1642.231689453125), w: 46.83233642578125, h: 47.7965087890625>,
    mask: None,
    score: PredictionScore: <value: 0.8943774700164795>,
    category: Category: <id: 0, name: item>>
[321.7101135253906, 1594.4351806640625, 368.5424499511719, 1642.231689453125]
torch.Size([1, 3, 64, 64])
ObjectPrediction<
    bbox: BoundingBox: <(416.4772033691406, 1741.5141296386719, 456.35943603515625, 1782.7207641601562), w: 39.882232666015625, h: 41.206634521484375>,
    mask: None,
    score: PredictionScore: <value: 

In [16]:
result.to_coco_annotations()[:1]

[{'image_id': None,
  'bbox': [361.2708435058594,
   1532.807876586914,
   50.584747314453125,
   52.10801696777344],
  'score': PredictionScore: <value: 0.8999278545379639>,
  'category_id': 2,
  'category_name': 'green',
  'segmentation': [],
  'iscrowd': 0,
  'area': 2635}]

In [23]:
# Export results
result.export_visuals(export_dir="demo_data/", hide_conf=True)
# Open the predicted image
processed_image = Image.open("demo_data/prediction_visual.png")
# Display the predicted image
processed_image.show()


(eog:3102278): Gtk-WARNING **: 15:08:11.157: cannot open display: 


In [18]:
from sahi.predict import predict

FOLDER_PATH = Path.home() / Path("BLUEBERRY_DATA/datasets/dataset-c834a7b7") / "images"

result = predict(
  model_type="ultralytics",
  model_path=str(MODEL_PATH),
  model_device=DEVICE,
  model_confidence_threshold=CONFIDENCE_THRESHOLD,
  source=str(FOLDER_PATH),
  slice_height=640,
  slice_width=640,
  overlap_height_ratio=0.2,
  overlap_width_ratio=0.2,
  postprocess_type="NMS", # defalt: GREEDYNMM
  postprocess_match_metric="IOU", #default:IOS
  postprocess_match_threshold=0.5
)

print(len(result.object_prediction_list))


There are 444 listed files in folder: images/


Performing inference on images:   0%|          | 0/444 [00:00<?, ?it/s]

Performing prediction on 12 slices.


Performing inference on images:   0%|          | 1/444 [00:00<03:12,  2.30it/s]

Prediction time is: 288.62 ms
Performing prediction on 12 slices.


...

Prediction time is: 203.51 ms
Performing prediction on 12 slices.


Performing inference on images:   3%|▎         | 14/444 [00:05<02:31,  2.83it/s]

Prediction time is: 213.90 ms
Performing prediction on 12 slices.


Performing inference on images:   3%|▎         | 14/444 [00:05<02:42,  2.65it/s]


KeyboardInterrupt: 

In [19]:
result.object_prediction_list[:2]

[ObjectPrediction<
     bbox: BoundingBox: <(361.2708435058594, 1532.807876586914, 411.8555908203125, 1584.9158935546875), w: 50.584747314453125, h: 52.10801696777344>,
     mask: None,
     score: PredictionScore: <value: PredictionScore: <value: 0.8999278545379639>>,
     category: Category: <id: 2, name: green>>,
 ObjectPrediction<
     bbox: BoundingBox: <(321.7101135253906, 1594.4351806640625, 368.5424499511719, 1642.231689453125), w: 46.83233642578125, h: 47.7965087890625>,
     mask: None,
     score: PredictionScore: <value: PredictionScore: <value: 0.8943774700164795>>,
     category: Category: <id: 2, name: green>>]

In [20]:
result.to_coco_annotations()[:1]

[{'image_id': None,
  'bbox': [361.2708435058594,
   1532.807876586914,
   50.584747314453125,
   52.10801696777344],
  'score': PredictionScore: <value: 0.8999278545379639>,
  'category_id': 2,
  'category_name': 'green',
  'segmentation': [],
  'iscrowd': 0,
  'area': 2635}]